## Smoke Validation

原 `check_lgbm_cuda_pipeline.py` 已合并到 notebook 顶部。
这一段使用 `NOTEBOOK_CONFIG`、`DATA_PATH` 和其它 notebook 全局变量完成轻量级 CUDA 训练链验证；后续单元直接复用这里生成的对象，不再重复训练。
            


In [ ]:
# 忽视警告，这个库是内置的，不需要安装
from pathlib import Path
import importlib.util
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

support_path_candidates = [
    Path("cuda_training_support.py"),
    Path("..") / "cuda_training_support.py",
]
SUPPORT_PATH = next((path.resolve() for path in support_path_candidates if path.exists()), None)
if SUPPORT_PATH is None:
    raise FileNotFoundError("找不到 cuda_training_support.py")

support_spec = importlib.util.spec_from_file_location("cuda_training_support", SUPPORT_PATH)
cuda_training_support = importlib.util.module_from_spec(support_spec)
sys.modules["cuda_training_support"] = cuda_training_support
support_spec.loader.exec_module(cuda_training_support)

NOTEBOOK_CONFIG = cuda_training_support.build_notebook_run_config()
DEFAULT_TARGET_COLUMN = cuda_training_support.DEFAULT_TARGET_COLUMN

# 兼容从项目根目录或 Notebook 所在目录启动内核的两种情况。
DATA_PATH = cuda_training_support.resolve_data_path(start_dir=Path.cwd())

print(
    cuda_training_support.format_notebook_run_summary(
        NOTEBOOK_CONFIG,
        data_path=DATA_PATH,
    )
)

# 如果内核从项目根目录启动，先移除本地 lightgbm 目录对官方包导入的遮蔽。
current_dir = Path.cwd().resolve()
if (current_dir / "lightgbm").is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

import lightgbm as lgb
from skopt import BayesSearchCV

random_seed = NOTEBOOK_CONFIG.random_seed

data = cuda_training_support.load_training_dataframe(
    data_path=DATA_PATH,
    random_seed=random_seed,
    run_mode=NOTEBOOK_CONFIG.run_mode,
    sample_size=NOTEBOOK_CONFIG.sample_size,
    target_column=DEFAULT_TARGET_COLUMN,
)
print(f"loaded_rows={len(data)}")
print("label_distribution=")
print(data[DEFAULT_TARGET_COLUMN].value_counts())

prepared = cuda_training_support.prepare_lightgbm_training_data(
    data,
    target_column=DEFAULT_TARGET_COLUMN,
    random_state=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
)
X_train = prepared["X_train"]
X_test = prepared["X_test"]
y_train = prepared["y_train"]
y_test = prepared["y_test"]
scaler = prepared["scaler"]

print(f"train_shape={X_train.shape}")
print(f"test_shape={X_test.shape}")
print("resampled_train_distribution=")
print(pd.Series(y_train).value_counts())

lgbm_version = cuda_training_support.validate_lightgbm_cuda_build(
    random_state=NOTEBOOK_CONFIG.random_seed,
)
print(f"lightgbm_version={lgbm_version}")
print("cuda_preflight=ok")

lgb_model = cuda_training_support.build_lgbm_classifier(
    random_state=NOTEBOOK_CONFIG.random_seed,
    model_n_jobs=NOTEBOOK_CONFIG.model_n_jobs,
)
search_spaces = cuda_training_support.get_lgbm_search_spaces(NOTEBOOK_CONFIG.run_mode)

bayes_search = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=search_spaces,
    n_iter=NOTEBOOK_CONFIG.bayes_n_iter,
    cv=NOTEBOOK_CONFIG.cv_folds,
    scoring="roc_auc",
    n_jobs=NOTEBOOK_CONFIG.search_n_jobs,
    verbose=1,
    random_state=NOTEBOOK_CONFIG.random_seed,
)

bayes_search.fit(X_train, y_train)
best_lgb = bayes_search.best_estimator_

y_test_pred = best_lgb.predict(X_test)
y_test_proba = best_lgb.predict_proba(X_test)[:, 1]

smoke_metrics = {
    "AUC": roc_auc_score(y_test, y_test_proba),
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall": recall_score(y_test, y_test_pred),
    "F1": f1_score(y_test, y_test_pred),
}
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
smoke_metrics["Specificity"] = tn / (tn + fp)
smoke_confusion_matrix = confusion_matrix(y_test, y_test_pred)

print(f"best_params={bayes_search.best_params_}")
for metric, value in smoke_metrics.items():
    print(f"{metric}={value:.6f}")
print("confusion_matrix=")
print(smoke_confusion_matrix)
    


In [ ]:
# 显示前几行数据，以确认数据已正确加载
(data.head(10))
            


In [ ]:
data['RETENTION_TIME'] = pd.to_numeric(data['RETENTION_TIME'], errors='coerce').astype('float64')
print("转换前 RETENTION_TIME 的示例值：")
print(data['RETENTION_TIME'].head())

non_numeric = pd.to_numeric(data['RETENTION_TIME'], errors='coerce').isna()
print()
print("非数值数据数量：", non_numeric.sum())
print("非数值数据示例：")
print(data[non_numeric]['RETENTION_TIME'].unique())
    


In [ ]:
print("样本总数:", len(data))
print("标签分布:")
data[DEFAULT_TARGET_COLUMN].value_counts()
            


In [ ]:
print("过采样方法:", "notebook_smote")
print("过采样后的训练集形状:", X_train.shape)
print("测试集形状:", X_test.shape)
print("过采样后的标签分布:", pd.Series(y_train).value_counts())
            


## LightGBM（CUDA-only）

- 顶部 smoke validation 单元已经完成数据加载、特征预处理、CUDA 预检和一次 BayesSearchCV 训练。
- 训练设备固定为 `cuda`，禁止 CPU fallback。
- `NOTEBOOK_CONFIG` 由 `build_notebook_run_config()` 生成，启动内核前可通过这些环境变量覆盖：`LGBM_NOTEBOOK_RUN_MODE`、`LGBM_SMOKE_SAMPLE_SIZE`、`LGBM_BAYES_N_ITER`、`LGBM_CV_FOLDS`、`LGBM_RANDOM_SEED`、`LGBM_TEST_SIZE`、`LGBM_MODEL_N_JOBS`、`LGBM_SEARCH_N_JOBS`。
- Notebook 默认使用 `smoke` 模式做轻量验证；设置 `LGBM_NOTEBOOK_RUN_MODE=full` 可切换为全量训练。
- 训练前会执行一次 CUDA 预检；如果当前 `lightgbm` 不是带 CUDA 的构建，会直接报错并停止。
- 官方当前不支持 Windows 上的 CUDA 版 LightGBM。需要把训练移动到 Linux 或 WSL2，并先执行：

```bash
pip uninstall -y lightgbm
pip install lightgbm --no-binary lightgbm --config-settings=cmake.define.USE_CUDA=ON
```
            


In [ ]:
print("LightGBM CUDA preflight:", lgbm_version)
print("最佳参数组合:", bayes_search.best_params_)
print("最佳验证集AUC:", bayes_search.best_score_)
for metric, value in smoke_metrics.items():
    print(f"Smoke {metric}: {value:.6f}")
print("测试集混淆矩阵:")
print(smoke_confusion_matrix)
            


In [ ]:
# 使用最佳参数训练模型
best_lgb = bayes_search.best_estimator_

# ============ 训练集评估 ============
y_train_pred = best_lgb.predict(X_train)
y_train_proba = best_lgb.predict_proba(X_train)[:, 1]

train_metrics = {
    'AUC': roc_auc_score(y_train, y_train_proba),
    'Accuracy': accuracy_score(y_train, y_train_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_train, y_train_pred),
    'Precision': precision_score(y_train, y_train_pred),
    'Recall': recall_score(y_train, y_train_pred),
    'F1': f1_score(y_train, y_train_pred)
}

tn, fp, fn, tp = confusion_matrix(y_train, y_train_pred).ravel()
train_metrics['Specificity'] = tn / (tn + fp)

# ============ 测试集评估 ============
y_test_pred = best_lgb.predict(X_test)
y_test_proba = best_lgb.predict_proba(X_test)[:, 1]

test_metrics = {
    'AUC': roc_auc_score(y_test, y_test_proba),
    'Accuracy': accuracy_score(y_test, y_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_test_pred),
    'Precision': precision_score(y_test, y_test_pred),
    'Recall': recall_score(y_test, y_test_pred),
    'F1': f1_score(y_test, y_test_pred)
}

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
test_metrics['Specificity'] = tn / (tn + fp)

print()
print("=== 训练集性能 ===")
for metric, value in train_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

print()
print("=== 测试集性能 ===")
for metric, value in test_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

print()
print("测试集混淆矩阵:")
print(confusion_matrix(y_test, y_test_pred))

lgb.plot_importance(best_lgb, max_num_features=20)
plt.tight_layout()
plt.show()
    
